# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a walk-through for loading, overviewing, and analyzing the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display dataset name and description
metadata = dataset.metadata
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("License:", metadata.license)

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data in _record sets_ (tables), with each record set referencing _fields_ (columns) via their `@id`.

Here, we'll enumerate all record sets, show their `@id`, and list their fields and field IDs.

In [ ]:
# List all record set entities and their fields
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets directly in the metadata. Attempting to extract via mlcroissant.")
    # Use the schema property to find record set entities
    record_sets = dataset.metadata.schema.get('@graph', [])
    # Filter for entities of cr:RecordSet type
    record_sets = [x for x in record_sets if x.get('@type', '') == 'cr:RecordSet']

if not record_sets:
    print("No record sets found in schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) else f
            print(f"    - {field_id}")

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis.

Use the record set and field `@id`s identified above. If there are multiple record sets, all will be loaded.

In [ ]:
# Identify all available record set IDs
record_set_ids = []
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
else:
    print("No record sets found; please check schema structure.")

dataframes = {}
for rs_id in record_set_ids:
    # Load records for each record set via their @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded RecordSet {rs_id} with shape {df.shape} and columns:")
    print(df.columns.tolist())
    print(df.head(3))

# Show columns for the first record set
if record_set_ids:
    exemplar_rs = record_set_ids[0]
    print("\nSample columns and data from:", exemplar_rs)
    print(dataframes[exemplar_rs].columns.tolist())
    display(dataframes[exemplar_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.

We'll select a numeric field, filter outliers, normalize distributions, and group results by a demographic field. All references are done with the field's `@id`.

In [ ]:
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[rs_id] if rs_id else pd.DataFrame()
print(f"Using record set: {rs_id}")

# Pick numeric field IDs; e.g. GAD-7 total score, PHQ-9 total score, etc.
numeric_fields = [col for col in df.columns if 'score' in col.lower() or df[col].dtype in [np.int64, np.float64]]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0] if len(df.columns) > 0 else None

print(f"Numeric field selected: {numeric_field_id}")

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if not filtered_df.empty:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find potential group fields for grouping, e.g. demographic fields
    possible_group_fields = ['gender', 'village', 'level_of_education', 'age']
    group_field = None
    for g in possible_group_fields:
        # Try to match the field id or subset
        for col in df.columns:
            if g in col.lower():
                group_field = col
                break
        if group_field:
            break
    
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No filtered records for threshold. Cannot perform further EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the (normalized) scores for the selected numeric field and, if available, group averages by demographic attributes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping was possible, plot group comparison
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,5))
        grouped_df.sort_values().plot.bar(color='teal')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`, referencing all data entities by their `@id`.

You can extend this analysis to examine other assessment scores, explore demographic patterns, and apply advanced statistical or machine learning methods to support public health research and policy development.

For further exploration, examine field definitions in the Croissant schema, link multiple record sets, and reference all columns via their unique IDs for robust, reproducible workflows.